# Настройки

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
import networkx as nx
import enum
from math import floor
import matplotlib.pyplot as plt
import random
import copy
import hashlib
import time
from einops import rearrange, repeat
import json
import pywt 
from utils import set_seed, resolve_out_channels
from layers import DTCWTLayer, IDTCWTLayer, DWTLayer, IWTLayer, TransformerBlock, \
                   SwinTransformerPairBlock, RestormerBlock, SimpleGate, IR_SHMABlock, InstanceEnhancementBatchNorm, SRMLayer, \
                   RCAB, ResGFMBlock
from train import train_model
from WaveletFusionNet import WaveletFusionNet

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

class OpType(enum.Enum):
    INPUT = 'input'
    CONV = 'conv'
    UPSAMPLE = 'upsample'
    BATCH_NORM = 'batch_norm'
    MAX_POOL = 'max_pool'
    LINEAR = 'linear'
    OUTPUT = 'output'
    FLATTEN = 'flatten'
    ADD = 'add'
    CONCAT = 'concat'
    DWT = 'dwt'
    IWT = 'iwt'
    DTCWT = 'dtcwt'
    IDTCWT = 'idtcwt'
    TRANSFORMER_BLOCK = 'transformer_block'
    GLOBAL_AVG_POOL = 'global_avg_pool'
    MULTIPLY = 'multiply'
    RELU = 'relu'
    PRELU = 'prelu'
    LEAKY_RELU = 'leaky_relu'
    ELU = 'elu'
    GELU = 'gelu'
    SILU = 'silu'
    SIGMOID = 'sigmoid'
    MISH = 'mish'
    LAYER_NORM = 'layer_norm'
    INSTANCE_NORM = 'instance_norm'
    GROUP_NORM = 'group_norm'
    SWIN_BLOCK = 'swin_block' 
    PIXEL_SHUFFLE = 'pixel_shuffle'
    PIXEL_UNSHUFFLE = 'pixel_unshuffle'
    RESTORMER_BLOCK = 'restormer_block'
    SIMPLE_GATE = 'simple_gate'
    IR_SHMA_BLOCK = 'ir_shma_block'
    IEBN = 'iebn'
    SRM = 'srm'
    RCAB = 'rcab'
    RES_GFM_BLOCK = 'res_gfm_block'


#обычные активации без параметров, не меняющие размерность выхода
ACTIVATION_MAP = {
    OpType.RELU: nn.ReLU,
    OpType.PRELU: nn.PReLU,
    OpType.LEAKY_RELU: nn.LeakyReLU,
    OpType.ELU: nn.ELU,
    OpType.GELU: nn.GELU,
    OpType.SILU: nn.SiLU,
    OpType.SIGMOID: nn.Sigmoid,
    OpType.MISH: nn.Mish,
}

PARAM_SPACE = {
    OpType.CONV: {
        'out_channels': ['same/8', 4, 'same/6', 6, 'same/4', 8, 'same/3', 12, 16, 'same/2', 24, 32, 
                         'same', 48, 'sameX2', 64, 'sameX3', 96, 'sameX4', 128,'sameX6', 192, 'sameX8', 256, 384, 512], 
        'kernel_size': [1, 3, 5, 7, 9, 11],
        'stride': [1, 2],
        'dilation': [1, 2, 3, 4],
        'groups': [1, 2, 4, 8, "depthwise"], # depthwise conv - особый случай, когда groups == in_channels
    },
    OpType.LINEAR: {
        'out_features': [64, 128, 256]
    },
    OpType.UPSAMPLE: {
        'scale_factor': [2.0, 4.0],
        'mode': ['nearest', 'bilinear']
    },
    OpType.PIXEL_SHUFFLE: {
        'upscale_factor': [2, 4]
    },
    OpType.PIXEL_UNSHUFFLE: {
        'downscale_factor': [2, 4]
    },
    OpType.DWT: {
        'wavelet_type': ['haar', 'db2', 'bior2.2'],
        'level': [1, 2],
    },
    OpType.IWT: {
        'wavelet_type': ['haar', 'db2', 'bior2.2'],
        'level': [1, 2],
    },
    OpType.DTCWT: {
        'level': [1, 2], 
    },
    OpType.IDTCWT: {
        'level': [1, 2],
    },
    OpType.TRANSFORMER_BLOCK: {
        'num_heads': [2, 4, 8],
        'mlp_ratio': [1.0, 2.0, 4.0],
    },
    OpType.SWIN_BLOCK: {
        'num_heads': [2, 4, 8],
        'window_size': [7],
        'mlp_ratio': [2.0, 4.0]
    },
    OpType.GROUP_NORM: {
        'num_groups': [2, 4, 8, 16, 32],
    },
    OpType.INSTANCE_NORM: {
        'affine': [True, False] # Обучаемые параметры (scale/shift)
    },
    OpType.LAYER_NORM: {
        'elementwise_affine': [True, False]
    },
    OpType.RESTORMER_BLOCK: {
        'num_heads': [1, 2, 4, 8],
        'ffn_expansion_factor': [2.66],
        'bias': [False, True],
        'LayerNorm_type': ['WithBias', 'BiasFree']
    },
    OpType.IR_SHMA_BLOCK: {
        'window_size': [8, 16, 32, 64],
        'num_chunks': [1, 2, 4],
        'mlp_ratio': [2.0, 3.0, 4.0]
    },
    OpType.RCAB: {
        'ca_reduction': [4, 8, 16],
        'bias': [True]
    },
    OpType.RES_GFM_BLOCK: {
        'inter_ratio': [1, 1.5, 2, 2.5],
        'bias': [True]
    }
}

MUTATION_BLOCKS = [
    # --- "Стандартный сверточный блок" ---
    [
        {'op_type': OpType.CONV, 'params': {'stride': 1}},
        {'op_type': OpType.RELU, 'params': {}},
    ],
    
    # --- "Классический U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.MAX_POOL, 'params': {}},
            {'op_type': OpType.CONV, 'params': {'stride': 1}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.UPSAMPLE, 'params': {}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        # Список индексов слоев, к которым добавить skip от входа блока
        "input_skip_targets": [4]
    },
    
    # --- "Pixel Shuffle U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.PIXEL_UNSHUFFLE, 'params': {'downscale_factor': 2}},
            {'op_type': OpType.CONV, 'params': {'stride': 1, 'kernel_size': 3, 'out_channels': 'same'}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.PIXEL_SHUFFLE, 'params': {'upscale_factor': 2}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
    {
        "layers": [
            {'op_type': OpType.PIXEL_UNSHUFFLE, 'params': {'downscale_factor': 4}},
            {'op_type': OpType.CONV, 'params': {'stride': 1, 'kernel_size': 3, 'out_channels': 'same'}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.PIXEL_SHUFFLE, 'params': {'upscale_factor': 4}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
    
    # --- "Wavelet U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.DWT, 'params': {'level': 1}},
            {'op_type': OpType.CONV, 'params': {'stride': 1}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.IWT, 'params': {'level': 1}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
    
    # --- "Double Tree Wavelet U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.DTCWT, 'params': {'level': 1}},
            {'op_type': OpType.CONV, 'params': {'stride': 1}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.IDTCWT, 'params': {'level': 1}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
    
    # --- "Double Wavelet U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.DWT, 'params': {'level': 2}},
            {'op_type': OpType.CONV, 'params': {'stride': 1}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.IWT, 'params': {'level': 2}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },

    # --- "Double Tree Wavelet U-Net спуск" ---
    {
        "layers": [
            {'op_type': OpType.DTCWT, 'params': {'level': 2}},
            {'op_type': OpType.CONV, 'params': {'stride': 1}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.IDTCWT, 'params': {'level': 2}},
            {'op_type': OpType.CONCAT, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
    
    # --- SE блок ---
    {
        "layers": [
            {'op_type': OpType.GLOBAL_AVG_POOL, 'params': {}},
            {'op_type': OpType.CONV, 'params': {'kernel_size': 1, 'out_channels': 'same/8'}},
            {'op_type': OpType.RELU, 'params': {}},
            {'op_type': OpType.CONV, 'params': {'kernel_size': 1, 'out_channels': 'sameX8'}},
            {'op_type': OpType.SIGMOID, 'params': {}},
            {'op_type': OpType.MULTIPLY, 'params': {}},
        ],
        "input_skip_targets": [5]
    },

    # --- Modern IR Depthwise Block ---
    {
        "layers": [
            {'op_type': OpType.LAYER_NORM, 'params': {'elementwise_affine': True}},
            
            {'op_type': OpType.CONV, 'params': {'stride': 1, 'groups': 'depthwise'}},
            {'op_type': OpType.GELU, 'params': {}},
            
            {'op_type': OpType.CONV, 'params': {'kernel_size': 1, 'stride': 1}},
            
            {'op_type': OpType.ADD, 'params': {}},
        ],
        "input_skip_targets": [4]
    },
]

PAIRED_OPS = {
    OpType.DWT: (OpType.IWT, 'level', 'level'),
    OpType.IWT: (OpType.DWT, 'level', 'level'),
    
    OpType.DTCWT: (OpType.IDTCWT, 'level', 'level'),
    OpType.IDTCWT: (OpType.DTCWT, 'level', 'level'),
    
    OpType.PIXEL_SHUFFLE: (OpType.PIXEL_UNSHUFFLE, 'upscale_factor', 'downscale_factor'),
    OpType.PIXEL_UNSHUFFLE: (OpType.PIXEL_SHUFFLE, 'downscale_factor', 'upscale_factor'),
}

BlocksWeights = [0.2, 0.04, 0.04, 0.02, 0.1, 0.1, 0.1, 0.1, 0.2, 0.1]


OpsToChoose = [OpType.CONV, OpType.RELU, OpType.PRELU, OpType.LEAKY_RELU, 
               OpType.ELU, OpType.GELU, OpType.SILU, OpType.MISH, 
               OpType.SIMPLE_GATE, OpType.TRANSFORMER_BLOCK, OpType.BATCH_NORM, OpType.GROUP_NORM, 
               OpType.INSTANCE_NORM, OpType.LAYER_NORM, OpType.SWIN_BLOCK, OpType.RESTORMER_BLOCK,
               OpType.IR_SHMA_BLOCK, OpType.IEBN, OpType.SRM, OpType.RCAB, 
               OpType.RES_GFM_BLOCK]

OpsWeights = [100, 20, 20, 20,
              20, 20, 20, 20, 
              30, 25, 20, 25, 
              35, 30, 30, 35,
              35, 25, 25, 30,
              30]

trainable_ops = [OpType.CONV, OpType.BATCH_NORM, OpType.LINEAR, OpType.TRANSFORMER_BLOCK, 
                 OpType.PRELU, OpType.LAYER_NORM, OpType.INSTANCE_NORM, OpType.GROUP_NORM, 
                 OpType.SWIN_BLOCK, OpType.RESTORMER_BLOCK, OpType.IR_SHMA_BLOCK, OpType.IEBN,
                 OpType.SRM, OpType.RCAB, OpType.RES_GFM_BLOCK]

MAX_CHANNELS = 1024
MAX_TOTAL_PARAMS = 100000000

In [ ]:
class ArchitectureGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.node_counter = 0
        
        # Каждая архитектура по умолчанию имеет вход и выход
        self.input_node = self.add_node(OpType.INPUT)
        self.output_node = self.add_node(OpType.OUTPUT)

    def _get_new_node_id(self):
        new_id = self.node_counter
        self.node_counter += 1
        return new_id

    def add_node(self, op_type: OpType, params: dict = None):
        node_id = self._get_new_node_id()
        self.graph.add_node(node_id, op_type=op_type, params=params or {})
        return node_id

    def add_edge(self, u_of_edge, v_of_edge):
        self.graph.add_edge(u_of_edge, v_of_edge)

    def _prune_branch_backwards(self, node_id):
        if not self.graph.has_node(node_id):
            return

        if node_id == self.input_node:
            return
        
        if self.graph.out_degree(node_id) > 0:
            return

        predecessors = list(self.graph.predecessors(node_id))
        
        self.graph.remove_node(node_id)
        
        for pred in predecessors:
            self._prune_branch_backwards(pred)

    def remove_node(self, node_id):
        if node_id in [self.input_node, self.output_node]:
            print("Warning: Cannot remove input/output nodes.")
            return

        predecessors = list(self.graph.predecessors(node_id))
        successors = list(self.graph.successors(node_id))
        
        self.graph.remove_node(node_id)

        if len(predecessors) <= 1:
            if predecessors:
                survivor = predecessors[0]
                for v in successors:
                    if not self.graph.has_edge(survivor, v):
                        self.graph.add_edge(survivor, v)
        
        else:
            survivor = random.choice(predecessors)
            
            doomed_nodes = [p for p in predecessors if p != survivor]
            
            for v in successors:
                if not self.graph.has_edge(survivor, v):
                    self.graph.add_edge(survivor, v)
            
            for doomed in doomed_nodes:
                self._prune_branch_backwards(doomed)

    def get_canonical_hash(self) -> str:
        try:
            sorted_nodes = list(nx.topological_sort(self.graph))
            
            node_hashes = {}

            for node_id in sorted_nodes:
                node_data = self.graph.nodes[node_id]
                
                predecessor_hashes = [
                    node_hashes[pred_id]
                    for pred_id in self.graph.predecessors(node_id)
                ]

                predecessor_hashes.sort()

                op_type = node_data['op_type'].name
                params = node_data.get('params', {})
                sorted_params = sorted(params.items())
                param_str = ";".join([f"{k}:{v}" for k, v in sorted_params])
                
                node_info_str = f"({op_type}:{param_str})"

                combined_info = f"{','.join(predecessor_hashes)}->{node_info_str}"
                
                current_hash = hashlib.sha256(combined_info.encode()).hexdigest()
                node_hashes[node_id] = current_hash

            return node_hashes[self.output_node]

        except nx.NetworkXUnfeasible:
            return "invalid_cycle_graph"

    def to_text(self, indent: int = 2) -> str:
        graph_representation = {
            "nodes": [],
            "edges": []
        }

        sorted_node_ids = sorted(self.graph.nodes())
        for node_id in sorted_node_ids:
            node_data = self.graph.nodes[node_id]
            graph_representation["nodes"].append({
                "id": node_id,
                "op_type": node_data['op_type'].name,
                "params": node_data.get('params', {})
            })

        sorted_edges = sorted(self.graph.edges())
        for u, v in sorted_edges:
            graph_representation["edges"].append([u, v])
            
        output_data = {
            "metadata": {
                "format_version": "1.0",
                "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
            },
            "graph": graph_representation
        }

        return json.dumps(output_data, indent=indent)

    @classmethod
    def from_text(cls, json_string: str) -> 'ArchitectureGraph':
        data = json.loads(json_string)
        graph_data = data['graph']

        arch = cls()
        arch.graph.clear()

        max_node_id = -1
        
        for node_info in graph_data['nodes']:
            node_id = node_info['id']
            op_type_str = node_info['op_type']
            params = node_info.get('params', {})
            
            try:
                op_type = OpType[op_type_str]
            except KeyError:
                raise ValueError(f"Unknown operation type '{op_type_str}' in JSON file.")

            arch.graph.add_node(node_id, op_type=op_type, params=params)
            
            if op_type == OpType.INPUT:
                arch.input_node = node_id
            elif op_type == OpType.OUTPUT:
                arch.output_node = node_id
            
            if node_id > max_node_id:
                max_node_id = node_id

        for u, v in graph_data['edges']:
            arch.add_edge(u, v)

        arch.node_counter = max_node_id + 1

        return arch

    def visualize(self, title="Architecture Graph"):
        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(self.graph)
        labels = {node: f"{data['op_type'].name}\nID: {node}" for node, data in self.graph.nodes(data=True)}
        nx.draw(self.graph, pos, labels=labels, with_labels=True, node_size=3000, node_color='skyblue', font_size=10, arrows=True)
        plt.title(title)
        plt.show()

    def export_to_image(self, filename="architecture.png"):
        try:
            from networkx.drawing.nx_agraph import to_agraph
        except ImportError:
            print("PyGraphviz not found. Please install it (see instructions above).")
            return

        A = to_agraph(self.graph)
        
        A.graph_attr.update(
            rankdir='LR',
            splines='polyline',
            nodesep='0.6',
            ranksep='1.2',
            fontname='Helvetica',
            bgcolor='white'
        )
        
        ABBREVIATIONS = {
            'out_channels': 'c',
            'kernel_size': 'k',
            'stride': 's',
            'dilation': 'd',
            'out_features': 'feat',
            'scale_factor': 'scale',
            'num_heads': 'heads',
            'mlp_ratio': 'mlp',
            'wavelet_type': 'wav',
            'level': 'lvl',
            'mode': 'mode',
            'window_size': 'win',
            'num_chunks': 'chunks',
            'num_groups': 'gr',
            'upscale_factor': 'up',
            'downscale_factor': 'down'
        }

        for node in A.nodes():
            nx_node_data = self.graph.nodes[int(node)]
            op_type = nx_node_data['op_type'].name
            params = nx_node_data.get('params', {})
            
            label_lines = [f"[{node}] {op_type}"]
            
            for param_key, param_val in sorted(params.items()):
                
                if param_key == 'groups':
                    if param_val == 'depthwise':
                        label_lines.append("Depthwise")
                    elif param_val > 1:
                        label_lines.append(f"g={param_val}")
                    continue 
                
                short_key = ABBREVIATIONS.get(param_key, param_key)
                
                val_str = str(param_val)
                if isinstance(param_val, float) and param_val.is_integer():
                    val_str = str(int(param_val))
                elif isinstance(param_val, float):
                     val_str = f"{param_val:.2f}"
                    
                label_lines.append(f"{short_key}={val_str}")

            full_label = "\n".join(label_lines)

            node.attr['label'] = full_label
            node.attr['shape'] = 'box'
            node.attr['style'] = 'filled, rounded'
            node.attr['fontsize'] = '10'
            node.attr['fontname'] = 'Helvetica'
            node.attr['penwidth'] = '1.5'
            
            if op_type == 'INPUT':
                node.attr['fillcolor'] = '#E0E0E0' 
                node.attr['shape'] = 'invhouse'    
            elif op_type == 'OUTPUT':
                node.attr['fillcolor'] = '#E0E0E0'
                node.attr['shape'] = 'house'       
            
            elif op_type == 'CONV':
                node.attr['fillcolor'] = '#A0CBE2'
            
            elif op_type == 'LINEAR':
                node.attr['fillcolor'] = '#FFBEBC'
            
            elif op_type in ['ADD', 'CONCAT', 'MULTIPLY', 'SIMPLE_GATE', 'FLATTEN']:
                node.attr['fillcolor'] = '#FFD7B5'
                node.attr['shape'] = 'ellipse'     
            
            elif op_type in ['TRANSFORMER_BLOCK', 'SWIN_BLOCK', 'RESTORMER_BLOCK', 
                             'IR_SHMA_BLOCK', 'RCAB', 'RES_GFM_BLOCK']:
                node.attr['fillcolor'] = '#FF9AA2'
            
            elif op_type in ['DWT', 'IWT', 'DTCWT', 'IDTCWT']:
                node.attr['fillcolor'] = '#E2F0CB'
            
            elif op_type in ['UPSAMPLE', 'PIXEL_SHUFFLE', 'PIXEL_UNSHUFFLE', 'MAX_POOL', 'GLOBAL_AVG_POOL']:
                node.attr['fillcolor'] = '#B5EAD7'
            
            elif op_type in ['RELU', 'PRELU', 'LEAKY_RELU', 'ELU', 'GELU', 'SILU', 'SIGMOID', 'MISH',
                             'BATCH_NORM', 'GROUP_NORM', 'INSTANCE_NORM', 'LAYER_NORM', 'IEBN', 'SRM']:
                node.attr['fillcolor'] = '#FFFAC8'
                node.attr['style'] = 'filled'      
                node.attr['height'] = '0.3'        
            
            else:
                node.attr['fillcolor'] = '#FFFFFF'

        print(f"Saving high-res graph to {filename}...")
        A.layout(prog='dot', args='-Gdpi=300') 
        A.draw(filename)
        
        import sys, os
        if sys.platform == 'darwin':
            os.system(f"open {filename}")

    def display_info(self):
        print("--- Nodes ---")
        sorted_node_ids = sorted(self.graph.nodes())
        
        for node_id in sorted_node_ids:
            node_data = self.graph.nodes[node_id]
            op_type_name = node_data['op_type'].name
            params = node_data.get('params')
            
            info_parts = [f"ID = {node_id}", op_type_name.lower()]
            
            if params:
                param_strings = [f"{key} = {value}" for key, value in params.items()]
                info_parts.extend(param_strings)
                
            print(", ".join(info_parts))
            
        print("\n--- Edges (Topological Order) ---")
        
        try:

            topologically_sorted_nodes = list(nx.topological_sort(self.graph))
            
            for u in topologically_sorted_nodes:
                successors = sorted(list(self.graph.successors(u)))
                for v in successors:
                    print(f"{u} -> {v}")

        except nx.NetworkXUnfeasible:
            print("Could not topologically sort the graph (it likely contains a cycle).")
            print("Falling back to sorting by source node ID:")
            sorted_edges = sorted(self.graph.edges())
            for u, v in sorted_edges:
                print(f"{u} -> {v}")

    def _try_sync_paired_node(self, source_node_id, param_name, new_value):
        source_op_type = self.graph.nodes[source_node_id]['op_type']
        
        if source_op_type not in PAIRED_OPS:
            return

        target_op_type, source_param_key, target_param_key = PAIRED_OPS[source_op_type]

        if param_name != source_param_key:
            return

        target_value = new_value
        if isinstance(new_value, float) and new_value.is_integer():
            target_value = int(new_value)
        elif isinstance(new_value, int) and source_op_type == OpType.PIXEL_UNSHUFFLE and target_op_type == OpType.UPSAMPLE:
             target_value = float(new_value)

        target_candidates = [
            n for n, d in self.graph.nodes(data=True) 
            if d['op_type'] == target_op_type
        ]

        if not target_candidates:
            return

        target_node_id = random.choice(target_candidates)
        
        target_node_data = self.graph.nodes[target_node_id]
        
        if target_op_type in PARAM_SPACE and target_param_key in PARAM_SPACE[target_op_type]:
            valid_values = PARAM_SPACE[target_op_type][target_param_key]
            if target_value not in valid_values:
                return

        target_node_data['params'][target_param_key] = target_value

    def mutate_add_node(self):
        if not self.graph.edges():
            print("Warning: No edges to mutate.")
            return
            
        edge_to_break = random.choice(list(self.graph.edges()))
        u, v = edge_to_break

        new_op_type = random.choices(OpsToChoose, OpsWeights, k=1)[0]
        params = {}
        if new_op_type in PARAM_SPACE:
            params = {key: random.choice(val) for key, val in PARAM_SPACE[new_op_type].items()}

        new_node_id = self.add_node(new_op_type, params)

        self.graph.remove_edge(u, v)
        self.add_edge(u, new_node_id)
        self.add_edge(new_node_id, v)
        print(f"Mutation: Added node {new_node_id} ({new_op_type.name}) between {u} and {v}")
    
    def mutate_add_block(self):
        if not self.graph.edges():
            print("Warning: No edges to mutate for block addition.")
            return
    
        edge_to_break = random.choice(list(self.graph.edges()))
        u, v = edge_to_break
        block_template = random.choices(MUTATION_BLOCKS, BlocksWeights, k=1)[0]
        
        if isinstance(block_template, list):
            self.graph.remove_edge(u, v)
            
            current_node_id = u
            for layer_template in block_template:
                op_type = layer_template['op_type']
                predefined_params = layer_template['params']
                
                final_params = {}
                if op_type in PARAM_SPACE:
                    final_params = {key: random.choice(val) for key, val in PARAM_SPACE[op_type].items()}
                final_params.update(predefined_params)
                
                new_node_id = self.add_node(op_type, final_params)
                self.add_edge(current_node_id, new_node_id)
                current_node_id = new_node_id
            
            self.add_edge(current_node_id, v)
        
        elif isinstance(block_template, dict):
            layers = block_template.get("layers", [])
            input_skip_targets = block_template.get("input_skip_targets", [])
            internal_edges = block_template.get("internal_edges", [])
            
            if not layers:
                print("Warning: Block template has no layers.")
                return
            
            self.graph.remove_edge(u, v)
            
            block_node_ids = []
            for layer_template in layers:
                op_type = layer_template['op_type']
                predefined_params = layer_template['params']
                
                final_params = {}
                if op_type in PARAM_SPACE:
                    final_params = {key: random.choice(val) for key, val in PARAM_SPACE[op_type].items()}
                final_params.update(predefined_params)
                
                new_node_id = self.add_node(op_type, final_params)
                block_node_ids.append(new_node_id)
            
            self.add_edge(u, block_node_ids[0])
            for i in range(len(block_node_ids) - 1):
                self.add_edge(block_node_ids[i], block_node_ids[i + 1])
            self.add_edge(block_node_ids[-1], v)
            
            for target_idx in input_skip_targets:
                if 0 <= target_idx < len(block_node_ids):
                    target_node = block_node_ids[target_idx]
                    if not self.graph.has_edge(u, target_node):
                        self.add_edge(u, target_node)
                        print(f"  Added input skip: {u} -> layer {target_idx} ({layers[target_idx]['op_type'].name})")
            
            for from_idx, to_idx in internal_edges:
                if (0 <= from_idx < len(block_node_ids) and 
                    0 <= to_idx < len(block_node_ids) and 
                    from_idx < to_idx):
                    from_node = block_node_ids[from_idx]
                    to_node = block_node_ids[to_idx]
                    if not self.graph.has_edge(from_node, to_node):
                        self.add_edge(from_node, to_node)
                        print(f"  Added internal skip: layer {from_idx} -> layer {to_idx}")

    def mutate_remove_node(self):
        removable_nodes = [
            node_id for node_id in self.graph.nodes()
            if node_id not in [self.input_node, self.output_node]
        ]

        if not removable_nodes:
            print("Warning: No removable nodes to mutate.")
            return

        node_to_remove = random.choice(removable_nodes)
        
        node_data = self.graph.nodes[node_to_remove]
        op_type_name = node_data['op_type'].name

        self.remove_node(node_to_remove)

    def mutate_change_params(self):
        mutable_nodes = [
            node_id for node_id, data in self.graph.nodes(data=True)
            if data['op_type'] in PARAM_SPACE
        ]
        if not mutable_nodes:
            print("Warning: No nodes with mutable parameters.")
            return

        node_to_mutate = random.choice(mutable_nodes)
        node_data = self.graph.nodes[node_to_mutate]
        op_type = node_data['op_type']
        
        param_to_change = random.choice(list(PARAM_SPACE[op_type].keys()))
        new_value = random.choice(PARAM_SPACE[op_type][param_to_change])
        
        old_value = node_data['params'].get(param_to_change)
        
        if old_value == new_value:
            options = [v for v in PARAM_SPACE[op_type][param_to_change] if v != old_value]
            if options:
                new_value = random.choice(options)
        
        node_data['params'][param_to_change] = new_value
        
        self._try_sync_paired_node(node_to_mutate, param_to_change, new_value)

    def mutate_step_params(self):
        mutable_nodes = [
            node_id for node_id, data in self.graph.nodes(data=True)
            if data['op_type'] in PARAM_SPACE
        ]
        
        if not mutable_nodes:
            print("Warning: No nodes with mutable parameters for step mutation.")
            return

        node_to_mutate = random.choice(mutable_nodes)
        node_data = self.graph.nodes[node_to_mutate]
        op_type = node_data['op_type']
        
        available_params = [
            p for p in PARAM_SPACE[op_type].keys() 
            if p in node_data['params']
        ]
        
        if not available_params:
            return

        param_to_change = random.choice(available_params)
        current_value = node_data['params'][param_to_change]
        possible_values = PARAM_SPACE[op_type][param_to_change]
        
        if len(possible_values) < 2:
            return

        try:
            current_idx = possible_values.index(current_value)
            
            if current_idx == 0:
                new_idx = 1
            elif current_idx == len(possible_values) - 1:
                new_idx = current_idx - 1
            else:
                step = random.choice([-1, 1])
                new_idx = current_idx + step
                
            new_value = possible_values[new_idx]
            
            node_data['params'][param_to_change] = new_value
            
            self._try_sync_paired_node(node_to_mutate, param_to_change, new_value)
            
        except ValueError:
            print(f"Warning: Current value {current_value} not found in PARAM_SPACE for {op_type.name}:{param_to_change}. Cannot step.")
            return

    def is_valid(self, input_shape: tuple) -> bool:
        if not nx.is_directed_acyclic_graph(self.graph):
            return False

        has_trainable_params = False
        for node_id in self.graph.nodes():
            node_data = self.graph.nodes[node_id]
            op_type = node_data['op_type']
            params = node_data.get('params', {})

            if op_type in trainable_ops:
                is_node_actually_trainable = True
                
                if op_type == OpType.LAYER_NORM:
                    if not params.get('elementwise_affine', True):
                        is_node_actually_trainable = False
                
                elif op_type == OpType.INSTANCE_NORM:
                    if not params.get('affine', False):
                        is_node_actually_trainable = False
                        
                elif op_type == OpType.GROUP_NORM:
                    if not params.get('affine', True):
                        is_node_actually_trainable = False
                
                if is_node_actually_trainable:
                    has_trainable_params = True
                    break
        
        if not has_trainable_params:
            return False
        
        try:
            sorted_nodes = list(nx.topological_sort(self.graph))
            if list(self.graph.predecessors(self.input_node)): return False
            if list(self.graph.successors(self.output_node)): return False

            shape_tracker = {self.input_node: input_shape}
            is_tensor_2d_plus = len(input_shape) > 1

            for node_id in sorted_nodes:
                if node_id == self.input_node:
                    continue
                
                predecessors = list(self.graph.predecessors(node_id))

                node_data = self.graph.nodes[node_id]
                op_type = node_data['op_type']

                if op_type == OpType.ADD:
                    if not predecessors: return False
                    pred_shapes = [shape_tracker.get(p) for p in predecessors]
                    if any(s is None for s in pred_shapes): return False
                    
                    first_shape = pred_shapes[0]
                    if not all(s == first_shape for s in pred_shapes[1:]):
                        return False
                    out_shape = first_shape
                
                elif op_type == OpType.MULTIPLY:
                    if not predecessors: return False
                    pred_shapes = [shape_tracker.get(p) for p in predecessors]
                    if any(s is None for s in pred_shapes): return False
                
                    if len(predecessors) == 1:
                        out_shape = pred_shapes[0]
                    else:
                        is_1d = any(len(s) == 1 for s in pred_shapes)
                        if is_1d and any(len(s) > 1 for s in pred_shapes):
                            return False
                
                        if is_1d:
                            target_len = max(s[0] for s in pred_shapes)
                            for shape in pred_shapes:
                                if not (shape[0] == target_len or shape[0] == 1):
                                    return False
                            out_shape = (target_len,)
                        else:
                            target_c = max(s[0] for s in pred_shapes)
                            target_h = max(s[1] for s in pred_shapes)
                            target_w = max(s[2] for s in pred_shapes)
                
                            for shape in pred_shapes:
                                c, h, w = shape
                                c_ok = (c == target_c or c == 1)
                                h_ok = (h == target_h or h == 1)
                                w_ok = (w == target_w or w == 1)
                                if not (c_ok and h_ok and w_ok):
                                    return False
                            
                            out_shape = (target_c, target_h, target_w)
                
                elif op_type == OpType.CONCAT:
                    if not predecessors: return False
                    pred_shapes = [shape_tracker.get(p) for p in predecessors]
                    if any(s is None for s in pred_shapes): return False
                    
                    first_shape = pred_shapes[0]
                    if len(predecessors) == 1:
                        out_shape = first_shape
                    else:
                        if not is_tensor_2d_plus: return False
                        if len(first_shape) < 3: return False
                        h_w_dims = (first_shape[1], first_shape[2])
                        if not all(len(s) >=3 and (s[1], s[2]) == h_w_dims for s in pred_shapes): return False
                        total_channels = sum(s[0] for s in pred_shapes)
                        out_shape = (total_channels, first_shape[1], first_shape[2])
                
                else:
                    if len(predecessors) != 1:
                        return False
                    
                    pred_id = predecessors[0]
                    if pred_id not in shape_tracker: return False
                    
                    in_shape = shape_tracker[pred_id]
                    params = node_data.get('params', {})
                    out_shape = in_shape

                if op_type == OpType.LINEAR and is_tensor_2d_plus:
                    flat_features = 1
                    for dim in in_shape: flat_features *= dim
                    in_shape = (flat_features,)
                    is_tensor_2d_plus = False

                if op_type == OpType.FLATTEN:
                    if is_tensor_2d_plus:
                        flat_features = 1
                        for dim in in_shape: flat_features *= dim
                        out_shape = (flat_features,)
                        is_tensor_2d_plus = False

                elif op_type == OpType.CONV:
                    if not is_tensor_2d_plus: return False
                    
                    if not all(k in params for k in ['out_channels', 'kernel_size']): return False
                    
                    in_channels, h_in, w_in = in_shape
                    
                    raw_out_param = params['out_channels']
                    out_channels = resolve_out_channels(in_channels, raw_out_param)
                    
                    groups_val = params.get('groups', 1)
                    
                    if groups_val == "depthwise":
                        out_channels = in_channels
                        groups_check = in_channels
                    else:
                        groups_check = groups_val
                    
                    if out_channels > MAX_CHANNELS:
                        return False
                    
                    if out_channels < 1: return False

                    if in_channels % groups_check != 0: return False
                    if out_channels % groups_check != 0: return False

                    stride = params.get('stride', 1)
                    dilation = params.get('dilation', 1)
                    kernel_size = params['kernel_size']
                    
                    if stride > 1:
                        k_eff = kernel_size + (kernel_size - 1) * (dilation - 1)
                        padding = (k_eff - 1) // 2
                    else:
                        padding = (kernel_size - 1) // 2 * dilation
                    
                    h_out = floor((h_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)
                    w_out = floor((w_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)

                    if h_out <= 0 or w_out <= 0: return False
                    
                    out_shape = (out_channels, h_out, w_out)

                elif op_type == OpType.SIMPLE_GATE:
                    if not is_tensor_2d_plus: return False
                    
                    if in_shape[0] % 2 != 0: 
                        return False
                    
                    out_shape = (in_shape[0] // 2, in_shape[1], in_shape[2])

                elif op_type == OpType.MAX_POOL:
                    if not is_tensor_2d_plus: return False
                    
                    kernel_size, stride = 2, 2
                    padding = 0
                    h_out = floor((in_shape[1] + 2 * padding - kernel_size) / stride + 1)
                    w_out = floor((in_shape[2] + 2 * padding - kernel_size) / stride + 1)
                    
                    if h_out <= 0 or w_out <= 0: return False
                    out_shape = (in_shape[0], h_out, w_out)

                elif op_type == OpType.UPSAMPLE:
                    if not is_tensor_2d_plus: return False
                    
                    scale_factor = params.get('scale_factor', 2.0)
                    mode = params.get('mode', 'nearest')
                    
                    if scale_factor <= 0: return False
                    
                    h_out = floor(in_shape[1] * scale_factor)
                    w_out = floor(in_shape[2] * scale_factor)
                    
                    if h_out <= 0 or w_out <= 0: return False
                    
                    out_shape = (in_shape[0], h_out, w_out)

                elif op_type == OpType.PIXEL_SHUFFLE:
                    if not is_tensor_2d_plus: return False
                    
                    upscale_factor = params.get('upscale_factor', 2)
                    squared_factor = upscale_factor * upscale_factor
                    
                    if in_shape[0] % squared_factor != 0:
                        return False
                        
                    out_channels = in_shape[0] // squared_factor
                    h_out = in_shape[1] * upscale_factor
                    w_out = in_shape[2] * upscale_factor
                    
                    out_shape = (out_channels, h_out, w_out)

                elif op_type == OpType.PIXEL_UNSHUFFLE:
                    if not is_tensor_2d_plus: return False
                    
                    downscale_factor = params.get('downscale_factor', 2)
                    
                    if in_shape[1] % downscale_factor != 0 or in_shape[2] % downscale_factor != 0:
                        return False
                        
                    squared_factor = downscale_factor * downscale_factor
                    out_channels = in_shape[0] * squared_factor
                    h_out = in_shape[1] // downscale_factor
                    w_out = in_shape[2] // downscale_factor
                    
                    if out_channels > MAX_CHANNELS:
                        return False

                    out_shape = (out_channels, h_out, w_out)

                elif op_type == OpType.DWT:
                    if not is_tensor_2d_plus: return False
                    level = params.get('level', 1)
                    if level < 1 or level > 2: return False
                    
                    h_in, w_in = in_shape[1], in_shape[2]
                    min_size = 2 ** level
                    if h_in % min_size != 0 or w_in % min_size != 0: return False
                    
                    out_shape = (in_shape[0] * (4 ** level), h_in // (2 ** level), w_in // (2 ** level))
                
                elif op_type == OpType.IWT:
                    if not is_tensor_2d_plus: return False
                    level = params.get('level', 1)
                    if level < 1 or level > 2: return False
                    
                    if in_shape[0] % (4 ** level) != 0: return False
                    
                    h_in, w_in = in_shape[1], in_shape[2]
                    out_shape = (in_shape[0] // (4 ** level), h_in * (2 ** level), w_in * (2 ** level))

                elif op_type == OpType.DTCWT:
                    if not is_tensor_2d_plus: return False
                    level = params.get('level', 1)
                    if level < 1 or level > 2: return False
                    
                    h_in, w_in = in_shape[1], in_shape[2]
                    min_size = 2 ** level
                    if h_in % min_size != 0 or w_in % min_size != 0: return False

                    required_min_size = 6 if level == 1 else 8
                    if h_in < required_min_size or w_in < required_min_size:
                        return False
                    
                    out_shape = (in_shape[0] * (8 ** level), h_in // (2 ** level), w_in // (2 ** level))

                elif op_type == OpType.IDTCWT:
                    if not is_tensor_2d_plus: return False
                    level = params.get('level', 1)
                    if level < 1 or level > 2: return False
                    
                    if in_shape[0] % (8 ** level) != 0: return False
                    
                    h_in, w_in = in_shape[1], in_shape[2]
                    out_shape = (in_shape[0] // (8 ** level), h_in * (2 ** level), w_in * (2 ** level))

                elif op_type == OpType.TRANSFORMER_BLOCK or op_type == OpType.SWIN_BLOCK:
                    if not is_tensor_2d_plus: return False
                    
                    embed_dim = in_shape[0]
                    num_heads = params.get('num_heads')
                    if num_heads is None: return False
                    if embed_dim % num_heads != 0: return False
                        
                    out_shape = in_shape

                elif op_type == OpType.RESTORMER_BLOCK:
                    if not is_tensor_2d_plus: return False
                    
                    in_channels = in_shape[0]
                    num_heads = params.get('num_heads', 1)
                    
                    if in_channels % num_heads != 0:
                        return False
                    
                    out_shape = in_shape

                elif op_type == OpType.IR_SHMA_BLOCK:
                    if not is_tensor_2d_plus: return False
                    
                    num_chunks = params.get('num_chunks', 1)
                    if in_shape[0] % num_chunks != 0:
                        return False
                        
                    out_shape = in_shape

                elif op_type == OpType.SRM:
                    if not is_tensor_2d_plus: return False
                    
                    out_shape = in_shape

                elif op_type == OpType.RCAB or op_type == OpType.RES_GFM_BLOCK:
                    if not is_tensor_2d_plus: return False
                    out_shape = in_shape

                elif op_type in ACTIVATION_MAP.keys() or op_type == OpType.BATCH_NORM or op_type == OpType.LAYER_NORM:
                    pass

                elif op_type == OpType.GROUP_NORM:
                    if len(in_shape) < 2: return False 
                    
                    num_channels = in_shape[0]
                    num_groups = params.get('num_groups', 2)
                    
                    if num_channels % num_groups != 0: 
                        return False
                        
                    if in_shape[1] <= 1 and in_shape[2] <= 1:
                        return False
                    
                    out_shape = in_shape

                elif op_type == OpType.INSTANCE_NORM:
                    if len(in_shape) < 2: 
                        return False
                    if in_shape[1] <= 1 and in_shape[2] <= 1:
                        return False
                    out_shape = in_shape

                elif op_type == OpType.IEBN:
                    if not is_tensor_2d_plus: 
                        return False
                    if in_shape[1] < 1 or in_shape[2] < 1:
                        return False
                    out_shape = in_shape

                elif op_type == OpType.GLOBAL_AVG_POOL:
                    if not is_tensor_2d_plus: return False
                    out_shape = (in_shape[0], 1, 1)
                
                elif op_type == OpType.LINEAR:
                    if is_tensor_2d_plus: return False
                    if 'out_features' not in params: return False
                    if len(in_shape) != 1: return False
                    if in_shape[0] == 0:
                        return False
                    out_shape = (params['out_features'],)

                shape_tracker[node_id] = out_shape


            final_predecessors = list(self.graph.predecessors(self.output_node))
    
            if len(final_predecessors) != 1:
                return False

            final_body_node = final_predecessors[0]

            if final_body_node not in shape_tracker:
                return False

            final_shape = shape_tracker[final_body_node]
            
            final_features = 1
            for dim in final_shape:
                final_features *= dim
            
            if final_features == 0:
                return False

            if final_shape != input_shape:
                return False

        except (KeyError, IndexError, TypeError, nx.NetworkXUnfeasible, nx.NetworkXError):
            return False
            
        return True

    def is_sane(self) -> bool:
        try:
            reachable_from_start = nx.descendants(self.graph, self.input_node)
            reachable_from_start.add(self.input_node)
            can_reach_end = nx.ancestors(self.graph, self.output_node)
            can_reach_end.add(self.output_node)
        except nx.NetworkXError:
            return False

        useful_nodes = reachable_from_start.intersection(can_reach_end)

        if set(self.graph.nodes()) != useful_nodes:
            return False

        forbidden_sequences = {
            OpType.FLATTEN, 
            OpType.BATCH_NORM, 
            OpType.ADD, 
            OpType.CONCAT,
            OpType.DWT,
            OpType.IWT
        }
        forbidden_sequences.update(ACTIVATION_MAP.keys())

        for u, v in self.graph.edges():
            op_type_u = self.graph.nodes[u]['op_type']
            op_type_v = self.graph.nodes[v]['op_type']

            if op_type_u == op_type_v and op_type_u in forbidden_sequences:
                return False

        return True

In [ ]:
class _GeneratedModel(nn.Module):
    def __init__(self, graph: nx.DiGraph, input_node_id, output_node_id, input_shape: tuple):
        super().__init__()
        
        self.graph = graph 
        self.input_node_id = input_node_id
        self.output_node_id = output_node_id

        self.sorted_nodes = list(nx.topological_sort(self.graph))

        self.layers = nn.ModuleDict()
        self._shape_tracker = {self.input_node_id: input_shape}
        
        is_tensor_2d_plus = len(input_shape) > 2
        
        for node_id in self.sorted_nodes:
            if node_id == self.input_node_id: 
                continue

            predecessors = list(self.graph.predecessors(node_id))
            if not predecessors: 
                continue

            node_data = self.graph.nodes[node_id]
            op_type = node_data['op_type']
            
            current_layer = None
            
            if op_type in [OpType.ADD, OpType.CONCAT, OpType.MULTIPLY]:
                pred_shapes = [self._shape_tracker[p] for p in predecessors]
    
                if op_type == OpType.ADD:
                    out_shape = pred_shapes[0]
                elif op_type == OpType.CONCAT:
                    first_shape = pred_shapes[0]
                    total_channels = sum(s[0] for s in pred_shapes)
                    out_shape = (total_channels, first_shape[1], first_shape[2])
                elif op_type == OpType.MULTIPLY:
                    if len(pred_shapes[0]) > 1:
                        target_c = max(s[0] for s in pred_shapes)
                        target_h = max(s[1] for s in pred_shapes)
                        target_w = max(s[2] for s in pred_shapes)
                        out_shape = (target_c, target_h, target_w)
                    else:
                        target_len = max(s[0] for s in pred_shapes)
                        out_shape = (target_len,)
            
            else:
                if len(predecessors) != 1:
                    raise ValueError(
                        f"Compilation Error: Node {node_id} (type: {op_type.name}) "
                        f"expects exactly 1 input, but received {len(predecessors)}. "
                        f"Predecessors: {predecessors}"
                    )
                    
                pred_id = predecessors[0]
                in_shape = self._shape_tracker[pred_id]
                params = node_data.get('params', {})
                
                if op_type == OpType.LINEAR and is_tensor_2d_plus:
                    flatten_key = f"internal_flatten_before_{node_id}"
                    self.layers[flatten_key] = nn.Flatten()
                    flat_features = 1
                    for dim in in_shape: flat_features *= dim
                    in_shape = (flat_features,)
                    is_tensor_2d_plus = False
    
                current_layer = None
                out_shape = in_shape
    
                if op_type == OpType.FLATTEN:
                    current_layer = nn.Flatten()
                    flat_features = 1
                    for dim in in_shape: flat_features *= dim
                    out_shape = (flat_features,)
                    is_tensor_2d_plus = False
    
                elif op_type == OpType.CONV:
                    in_channels, h_in, w_in = in_shape
                    
                    groups_val = params.get('groups', 1)
                    
                    if groups_val == "depthwise":
                        groups = in_channels
                        out_channels = in_channels 
                    else:
                        groups = groups_val
                        raw_out_param = params['out_channels']
                        out_channels = resolve_out_channels(in_channels, raw_out_param)

                    if out_channels > MAX_CHANNELS:
                        raise ValueError(f"Security stop: Layer {node_id} requests {out_channels} channels (MAX={MAX_CHANNELS})")
    
                    stride = params.get('stride', 1)
                    dilation = params.get('dilation', 1)
                    kernel_size = params['kernel_size']
    
                    padding_value = 'same'
                    if stride > 1:
                        k_eff = kernel_size + (kernel_size - 1) * (dilation - 1)
                        padding_value = (k_eff - 1) // 2
                    
                    current_layer = nn.Conv2d(
                        in_channels=in_channels, 
                        out_channels=out_channels, 
                        kernel_size=kernel_size, 
                        stride=stride, 
                        padding=padding_value, 
                        groups=groups,
                        dilation=dilation
                    )
                    
                    if stride > 1:
                        k_eff = kernel_size + (kernel_size - 1) * (dilation - 1)
                        padding = (k_eff - 1) // 2
                    else:
                        padding = (kernel_size - 1) // 2 * dilation
                    
                    h_out = floor((h_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)
                    w_out = floor((w_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)
                    out_shape = (out_channels, h_out, w_out)

                elif op_type in ACTIVATION_MAP:
                    current_layer = ACTIVATION_MAP[op_type]()

                elif op_type == OpType.SIMPLE_GATE:
                    current_layer = SimpleGate()
                    out_shape = (in_shape[0] // 2, in_shape[1], in_shape[2])
                    
                elif op_type == OpType.BATCH_NORM:
                    num_features = in_shape[0]
                    current_layer = nn.BatchNorm2d(num_features) if is_tensor_2d_plus else nn.BatchNorm1d(num_features)

                elif op_type == OpType.GROUP_NORM:
                    num_channels = in_shape[0]
                    num_groups = params['num_groups']
                    current_layer = nn.GroupNorm(num_groups=num_groups, num_channels=num_channels)

                elif op_type == OpType.INSTANCE_NORM:
                    num_features = in_shape[0]
                    affine = params.get('affine', False)
                    if is_tensor_2d_plus:
                        current_layer = nn.InstanceNorm2d(num_features, affine=affine)
                    else:
                        current_layer = nn.InstanceNorm1d(num_features, affine=affine)

                elif op_type == OpType.IEBN:                    
                    num_features = in_shape[0]
                    current_layer = InstanceEnhancementBatchNorm(num_features, affine=True)
                    out_shape = in_shape

                elif op_type == OpType.LAYER_NORM:
                    elementwise_affine = params.get('elementwise_affine', True)
                    
                    if is_tensor_2d_plus:
                        current_layer = nn.GroupNorm(num_groups=1, num_channels=in_shape[0], affine=elementwise_affine)
                    else:
                        current_layer = nn.LayerNorm(in_shape, elementwise_affine=elementwise_affine)

                elif op_type == OpType.MAX_POOL:
                    kernel_size, stride = 2, 2
                    current_layer = nn.MaxPool2d(kernel_size=kernel_size, stride=stride)
                    padding = 0
                    h_out = floor((in_shape[1] + 2 * padding - kernel_size) / stride + 1)
                    w_out = floor((in_shape[2] + 2 * padding - kernel_size) / stride + 1)
                    out_shape = (in_shape[0], h_out, w_out)

                elif op_type == OpType.UPSAMPLE:
                    scale_factor = params.get('scale_factor', 2.0)
                    mode = params.get('mode', 'nearest')
                    
                    current_layer = nn.Upsample(scale_factor=scale_factor, mode=mode)
                    if mode == 'bilinear':
                        current_layer.align_corners = False
                    
                    h_out = floor(in_shape[1] * scale_factor)
                    w_out = floor(in_shape[2] * scale_factor)
                    out_shape = (in_shape[0], h_out, w_out)

                elif op_type == OpType.PIXEL_SHUFFLE:
                    factor = params.get('upscale_factor', 2)
                    current_layer = nn.PixelShuffle(upscale_factor=factor)

                    sq = factor * factor
                    out_shape = (in_shape[0] // sq, in_shape[1] * factor, in_shape[2] * factor)

                elif op_type == OpType.PIXEL_UNSHUFFLE:
                    factor = params.get('downscale_factor', 2)
                    current_layer = nn.PixelUnshuffle(downscale_factor=factor)
                    
                    sq = factor * factor
                    out_shape = (in_shape[0] * sq, in_shape[1] // factor, in_shape[2] // factor)

                elif op_type == OpType.DWT:
                    in_channels = in_shape[0]
                    level = params.get('level', 1)
                    wave_type = params.get('wavelet_type', 'haar')
                    
                    current_layer = DWTLayer(wave=wave_type, level=level)
                    
                    out_shape = (in_channels * (4 ** level), in_shape[1] // (2 ** level), in_shape[2] // (2 ** level))
                
                elif op_type == OpType.IWT:
                    in_channels = in_shape[0]
                    level = params.get('level', 1)
                    wave_type = params.get('wavelet_type', 'haar')
                    
                    current_layer = IWTLayer(wave=wave_type, level=level)
                    
                    out_shape = (in_channels // (4 ** level), in_shape[1] * (2 ** level), in_shape[2] * (2 ** level))

                elif op_type == OpType.DTCWT:
                    level = params.get('level', 1)
                    current_layer = DTCWTLayer(level=level)
                    in_ch = in_shape[0]
                    out_shape = (in_ch * (8 ** level), in_shape[1] // (2 ** level), in_shape[2] // (2 ** level))

                elif op_type == OpType.IDTCWT:
                    level = params.get('level', 1)
                    current_layer = IDTCWTLayer(level=level)
                    in_ch = in_shape[0]
                    out_shape = (in_ch // (8 ** level), in_shape[1] * (2 ** level), in_shape[2] * (2 ** level))
                
                elif op_type == OpType.TRANSFORMER_BLOCK:
                    embed_dim = in_shape[0]
                    num_heads = params['num_heads']
                    mlp_ratio = params.get('mlp_ratio', 4.0)
                    
                    current_layer = TransformerBlock(
                        dim=embed_dim,
                        num_heads=num_heads,
                        mlp_ratio=mlp_ratio
                    )
                    out_shape = in_shape

                elif op_type == OpType.SWIN_BLOCK:
                    dim = in_shape[0]
                    num_heads = params['num_heads']
                    window_size = params.get('window_size', 7)
                    mlp_ratio = params.get('mlp_ratio', 4.0)
                    
                    current_h, current_w = in_shape[1], in_shape[2]
                    
                    actual_window_size = window_size
                    if current_h < window_size or current_w < window_size:
                        actual_window_size = min(current_h, current_w)
                        actual_window_size = max(1, actual_window_size)

                    current_layer = SwinTransformerPairBlock(
                        dim=dim,
                        input_resolution=(current_h, current_w),
                        num_heads=num_heads,
                        window_size=actual_window_size,
                        mlp_ratio=mlp_ratio
                    )
                    out_shape = in_shape

                elif op_type == OpType.RESTORMER_BLOCK:
                    dim = in_shape[0]
                    
                    num_heads = params['num_heads']
                    ffn_expansion_factor = params.get('ffn_expansion_factor', 2.66)
                    bias = params.get('bias', False)
                    norm_type = params.get('LayerNorm_type', 'WithBias')
                    
                    current_layer = RestormerBlock(
                        dim=dim,
                        num_heads=num_heads,
                        ffn_expansion_factor=ffn_expansion_factor,
                        bias=bias,
                        LayerNorm_type=norm_type
                    )
                    
                    out_shape = in_shape

                elif op_type == OpType.IR_SHMA_BLOCK:
                    dim = in_shape[0]

                    window_size = params.get('window_size', 16)
                    num_chunks = params.get('num_chunks', 1)
                    mlp_ratio = params.get('mlp_ratio', 2.0)
                    
                    current_layer = IR_SHMABlock(
                        dim=dim,
                        window_size=window_size,
                        num_chunks=num_chunks,
                        mlp_ratio=mlp_ratio,
                        drop_path=0.0
                    )
                    
                    out_shape = in_shape

                elif op_type == OpType.SRM:
                    in_channels = in_shape[0]
                    current_layer = SRMLayer(channels=in_channels)
                    out_shape = in_shape

                elif op_type == OpType.RCAB:
                    num_feat = in_shape[0]
                    ca_reduction = params.get('ca_reduction', 8)
                    bias = params.get('bias', True)
                    
                    current_layer = RCAB(
                        num_feat=num_feat, 
                        ca_reduction=ca_reduction, 
                        bias=bias
                    )
                    out_shape = in_shape

                elif op_type == OpType.RES_GFM_BLOCK:
                    num_feat = in_shape[0]
                    inter_ratio = params.get('inter_ratio', 1.5)
                    bias = params.get('bias', True)
                    
                    current_layer = ResGFMBlock(
                        num_feat=num_feat, 
                        inter_ratio=inter_ratio, 
                        bias=bias
                    )
                    out_shape = in_shape

                elif op_type == OpType.GLOBAL_AVG_POOL:
                    current_layer = nn.AdaptiveAvgPool2d((1, 1))
                    out_shape = (in_shape[0], 1, 1)

                
                elif op_type == OpType.LINEAR:
                    current_layer = nn.Linear(in_shape[0], params['out_features'])
                    out_shape = (params['out_features'],)

            if current_layer:
                self.layers[str(node_id)] = current_layer
            
            self._shape_tracker[node_id] = out_shape

        final_predecessors = list(self.graph.predecessors(self.output_node_id))
        if len(final_predecessors) != 1:
            raise ValueError(
                f"Compilation Error: The output node ({self.output_node_id}) must have exactly one "
                f"predecessor, but it has {len(final_predecessors)}. Predecessors: {final_predecessors}. "
                "A merge layer (e.g., ADD, CONCAT) might be required before the output."
            )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        outputs = {self.input_node_id: x}
        for node_id in self.sorted_nodes:
            if node_id == self.input_node_id: 
                continue

            predecessors = list(self.graph.predecessors(node_id))
            op_type = self.graph.nodes[node_id]['op_type']
            
            input_tensors = [outputs[pred_id] for pred_id in predecessors]

            if op_type == OpType.ADD:
                input_tensor = torch.stack(input_tensors, dim=0).sum(dim=0)
            elif op_type == OpType.CONCAT:
                input_tensor = torch.cat(input_tensors, dim=1)
            elif op_type == OpType.MULTIPLY:
                result_tensor = input_tensors[0]
                for i in range(1, len(input_tensors)):
                    result_tensor = result_tensor * input_tensors[i]
                input_tensor = result_tensor
            else:
                input_tensor = input_tensors[0]
            
            flatten_key = f"internal_flatten_before_{node_id}"
            if flatten_key in self.layers:
                input_tensor = self.layers[flatten_key](input_tensor)
                
            if str(node_id) in self.layers:
                output_tensor = self.layers[str(node_id)](input_tensor)
            else:
                output_tensor = input_tensor
                
            outputs[node_id] = output_tensor
        
        final_body_node = list(self.graph.predecessors(self.output_node_id))[0]
        
        final_output = outputs[final_body_node]
        
        return final_output

def build_model_from_graph(arch_graph: ArchitectureGraph, input_shape: tuple) -> _GeneratedModel:
    model = _GeneratedModel(
        graph=arch_graph.graph,
        input_node_id=arch_graph.input_node,
        output_node_id=arch_graph.output_node,
        input_shape=input_shape
    )
    
    total_params = sum(p.numel() for p in model.parameters())

    if total_params > MAX_TOTAL_PARAMS: 
        raise ValueError(f"Model too large: {total_params} params")
        
    return model 

In [ ]:
import torch
import torch.nn as nn
import networkx as nx
from math import floor
import inspect

class PyTorchCodeGenerator:
    def __init__(self, graph_obj, input_shape: tuple):
        self.graph = graph_obj.graph
        self.input_node_id = graph_obj.input_node
        self.output_node_id = graph_obj.output_node
        self.input_shape = input_shape
        
        self.sorted_nodes = list(nx.topological_sort(self.graph))
        self.shape_tracker = {self.input_node_id: input_shape}
        
        self.init_lines = []
        self.forward_lines = []
        self.imports = set()

        self.imports.add("import torch")
        self.imports.add("import torch.nn as nn")
        self.imports.add("import torch.nn.functional as F")

    def _get_padding(self, h_in, w_in, stride, dilation, kernel_size):
        if stride > 1:
            k_eff = kernel_size + (kernel_size - 1) * (dilation - 1)
            padding = (k_eff - 1) // 2
        else:
            padding = (kernel_size - 1) // 2 * dilation
        
        h_out = floor((h_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)
        w_out = floor((w_in + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)
        return padding, h_out, w_out

    def generate(self, class_name="BestModel", file_path="generated_model.py"):
        is_tensor_2d_plus = len(self.input_shape) > 2
        
        self.forward_lines.append(f"x_{self.input_node_id} = x")

        for node_id in self.sorted_nodes:
            if node_id == self.input_node_id:
                continue

            predecessors = list(self.graph.predecessors(node_id))
            if not predecessors:
                continue

            node_data = self.graph.nodes[node_id]
            op_type = node_data['op_type']
            params = node_data.get('params', {})
            
            if op_type in [OpType.ADD, OpType.CONCAT, OpType.MULTIPLY]:
                pred_shapes = [self.shape_tracker[p] for p in predecessors]
                input_vars = [f"x_{p}" for p in predecessors]
                
                if op_type == OpType.ADD:
                    out_shape = pred_shapes[0]
                    line = f"x_{node_id} = " + " + ".join(input_vars)
                    self.forward_lines.append(line)
                    
                elif op_type == OpType.CONCAT:
                    first_shape = pred_shapes[0]
                    total_channels = sum(s[0] for s in pred_shapes)
                    out_shape = (total_channels, first_shape[1], first_shape[2])
                    line = f"x_{node_id} = torch.cat([{', '.join(input_vars)}], dim=1)"
                    self.forward_lines.append(line)
                    
                elif op_type == OpType.MULTIPLY:
                    if len(pred_shapes[0]) > 1:
                        target_c = max(s[0] for s in pred_shapes)
                        target_h = max(s[1] for s in pred_shapes)
                        target_w = max(s[2] for s in pred_shapes)
                        out_shape = (target_c, target_h, target_w)
                    else:
                        target_len = max(s[0] for s in pred_shapes)
                        out_shape = (target_len,)
                    
                    line = f"x_{node_id} = {input_vars[0]}"
                    for i in range(1, len(input_vars)):
                        line += f" * {input_vars[i]}"
                    self.forward_lines.append(line)
                
                self.shape_tracker[node_id] = out_shape
                continue

            pred_id = predecessors[0]
            in_shape = self.shape_tracker[pred_id]
            input_var = f"x_{pred_id}"
            
            layer_name = f"self.layer_{node_id}"
            layer_def = None
            use_layer_in_forward = True
            
            if op_type == OpType.LINEAR and is_tensor_2d_plus:
                flatten_name = f"self.flatten_{node_id}"
                self.init_lines.append(f"{flatten_name} = nn.Flatten()")
                
                flattened_var = f"x_{pred_id}_flat"
                self.forward_lines.append(f"{flattened_var} = {flatten_name}({input_var})")
                input_var = flattened_var
                
                flat_features = 1
                for dim in in_shape: flat_features *= dim
                in_shape = (flat_features,)
                is_tensor_2d_plus = False

            out_shape = in_shape
            
            if op_type == OpType.FLATTEN:
                layer_def = "nn.Flatten()"
                flat_features = 1
                for dim in in_shape: flat_features *= dim
                out_shape = (flat_features,)
                is_tensor_2d_plus = False
            
            elif op_type == OpType.CONV:
                in_channels, h_in, w_in = in_shape
                
                groups_val = params.get('groups', 1)
                
                if groups_val == "depthwise":
                    groups = in_channels
                    out_channels = in_channels 
                else:
                    groups = groups_val
                    raw_out_param = params['out_channels']
                    out_channels = resolve_out_channels(in_channels, raw_out_param)

                stride = params.get('stride', 1)
                dilation = params.get('dilation', 1)
                kernel_size = params['kernel_size']
                
                padding, h_out, w_out = self._get_padding(h_in, w_in, stride, dilation, kernel_size)
                out_shape = (out_channels, h_out, w_out)
                
                layer_def = (f"nn.Conv2d(in_channels={in_channels}, out_channels={out_channels}, "
                             f"kernel_size={kernel_size}, stride={stride}, padding={padding}, "
                             f"dilation={dilation}, groups={groups})")
            
            elif op_type in ACTIVATION_MAP:
                act_class = ACTIVATION_MAP[op_type].__name__
                layer_def = f"nn.{act_class}()"
                
            elif op_type == OpType.SIMPLE_GATE:
                self.imports.add("from layers import SimpleGate")
                layer_def = "SimpleGate()"
                out_shape = (in_shape[0] // 2, in_shape[1], in_shape[2])
                
            elif op_type == OpType.BATCH_NORM:
                num_features = in_shape[0]
                bn_type = "nn.BatchNorm2d" if is_tensor_2d_plus else "nn.BatchNorm1d"
                layer_def = f"{bn_type}(num_features={num_features})"
            
            elif op_type == OpType.GROUP_NORM:
                num_channels = in_shape[0]
                num_groups = params['num_groups']
                layer_def = f"nn.GroupNorm(num_groups={num_groups}, num_channels={num_channels})"
                
            elif op_type == OpType.INSTANCE_NORM:
                num_features = in_shape[0]
                affine = params.get('affine', False)
                in_type = "nn.InstanceNorm2d" if is_tensor_2d_plus else "nn.InstanceNorm1d"
                layer_def = f"{in_type}(num_features={num_features}, affine={affine})"
                
            elif op_type == OpType.IEBN:
                self.imports.add("from layers import InstanceEnhancementBatchNorm")
                layer_def = f"InstanceEnhancementBatchNorm(num_features={in_shape[0]}, affine=True)"
                
            elif op_type == OpType.LAYER_NORM:
                elementwise_affine = params.get('elementwise_affine', True)
                if is_tensor_2d_plus:
                    layer_def = f"nn.GroupNorm(num_groups=1, num_channels={in_shape[0]}, affine={elementwise_affine})"
                else:
                    layer_def = f"nn.LayerNorm(normalized_shape={in_shape}, elementwise_affine={elementwise_affine})"

            elif op_type == OpType.MAX_POOL:
                padding, h_out, w_out = self._get_padding(in_shape[1], in_shape[2], 2, 1, 2)
                h_out = floor((in_shape[1] - 2) / 2 + 1)
                w_out = floor((in_shape[2] - 2) / 2 + 1)
                out_shape = (in_shape[0], h_out, w_out)
                layer_def = "nn.MaxPool2d(kernel_size=2, stride=2)"

            elif op_type == OpType.UPSAMPLE:
                scale = params.get('scale_factor', 2.0)
                mode = params.get('mode', 'nearest')
                align = "None"
                if mode == 'bilinear': align = "False"
                layer_def = f"nn.Upsample(scale_factor={scale}, mode='{mode}', align_corners={align})"
                
                h_out = floor(in_shape[1] * scale)
                w_out = floor(in_shape[2] * scale)
                out_shape = (in_shape[0], h_out, w_out)

            elif op_type == OpType.PIXEL_SHUFFLE:
                factor = params.get('upscale_factor', 2)
                layer_def = f"nn.PixelShuffle(upscale_factor={factor})"
                sq = factor * factor
                out_shape = (in_shape[0] // sq, in_shape[1] * factor, in_shape[2] * factor)

            elif op_type == OpType.PIXEL_UNSHUFFLE:
                factor = params.get('downscale_factor', 2)
                layer_def = f"nn.PixelUnshuffle(downscale_factor={factor})"
                sq = factor * factor
                out_shape = (in_shape[0] * sq, in_shape[1] // factor, in_shape[2] // factor)

            elif op_type == OpType.DWT:
                self.imports.add("from layers import DWTLayer")
                level = params.get('level', 1)
                wtype = params.get('wavelet_type', 'haar')
                layer_def = f"DWTLayer(wave='{wtype}', level={level})"
                out_shape = (in_shape[0] * (4 ** level), in_shape[1] // (2 ** level), in_shape[2] // (2 ** level))
            
            elif op_type == OpType.IWT:
                self.imports.add("from layers import IWTLayer")
                level = params.get('level', 1)
                wtype = params.get('wavelet_type', 'haar')
                layer_def = f"IWTLayer(wave='{wtype}', level={level})"
                out_shape = (in_shape[0] // (4 ** level), in_shape[1] * (2 ** level), in_shape[2] * (2 ** level))

            elif op_type == OpType.DTCWT:
                self.imports.add("from layers import DTCWTLayer")
                level = params.get('level', 1)
                layer_def = f"DTCWTLayer(level={level})"
                out_shape = (in_shape[0] * (8 ** level), in_shape[1] // (2 ** level), in_shape[2] // (2 ** level))

            elif op_type == OpType.IDTCWT:
                self.imports.add("from layers import IDTCWTLayer")
                level = params.get('level', 1)
                layer_def = f"IDTCWTLayer(level={level})"
                out_shape = (in_shape[0] // (8 ** level), in_shape[1] * (2 ** level), in_shape[2] * (2 ** level))

            elif op_type == OpType.TRANSFORMER_BLOCK:
                self.imports.add("from layers import TransformerBlock")
                layer_def = (f"TransformerBlock(dim={in_shape[0]}, "
                             f"num_heads={params['num_heads']}, "
                             f"mlp_ratio={params.get('mlp_ratio', 4.0)})")

            elif op_type == OpType.SWIN_BLOCK:
                self.imports.add("from layers import SwinTransformerPairBlock")
                h, w = in_shape[1], in_shape[2]
                ws = params.get('window_size', 7)
                actual_ws = ws
                if h < ws or w < ws:
                    actual_ws = min(h, w)
                    actual_ws = max(1, actual_ws)
                
                layer_def = (f"SwinTransformerPairBlock(dim={in_shape[0]}, "
                             f"input_resolution=({h}, {w}), "
                             f"num_heads={params['num_heads']}, "
                             f"window_size={actual_ws}, "
                             f"mlp_ratio={params.get('mlp_ratio', 4.0)})")

            elif op_type == OpType.RESTORMER_BLOCK:
                self.imports.add("from layers import RestormerBlock")
                layer_def = (f"RestormerBlock(dim={in_shape[0]}, "
                             f"num_heads={params['num_heads']}, "
                             f"ffn_expansion_factor={params.get('ffn_expansion_factor', 2.66)}, "
                             f"bias={params.get('bias', False)}, "
                             f"LayerNorm_type='{params.get('LayerNorm_type', 'WithBias')}')")

            elif op_type == OpType.IR_SHMA_BLOCK:
                self.imports.add("from layers import IR_SHMABlock")
                layer_def = (f"IR_SHMABlock(dim={in_shape[0]}, "
                             f"window_size={params.get('window_size', 16)}, "
                             f"num_chunks={params.get('num_chunks', 1)}, "
                             f"mlp_ratio={params.get('mlp_ratio', 2.0)}, "
                             f"drop_path=0.0)")

            elif op_type == OpType.SRM:
                self.imports.add("from layers import SRMLayer")
                layer_def = f"SRMLayer(channels={in_shape[0]})"
                
            elif op_type == OpType.RCAB:
                self.imports.add("from layers import RCAB")
                layer_def = (f"RCAB(num_feat={in_shape[0]}, "
                             f"ca_reduction={params.get('ca_reduction', 8)}, "
                             f"bias={params.get('bias', True)})")
                             
            elif op_type == OpType.RES_GFM_BLOCK:
                self.imports.add("from layers import ResGFMBlock")
                layer_def = (f"ResGFMBlock(num_feat={in_shape[0]}, "
                             f"inter_ratio={params.get('inter_ratio', 1.5)}, "
                             f"bias={params.get('bias', True)})")

            elif op_type == OpType.GLOBAL_AVG_POOL:
                layer_def = "nn.AdaptiveAvgPool2d((1, 1))"
                out_shape = (in_shape[0], 1, 1)

            elif op_type == OpType.LINEAR:
                layer_def = f"nn.Linear(in_features={in_shape[0]}, out_features={params['out_features']})"
                out_shape = (params['out_features'],)

            if layer_def:
                self.init_lines.append(f"{layer_name} = {layer_def}")
                self.forward_lines.append(f"x_{node_id} = {layer_name}({input_var})")
            else:
                self.forward_lines.append(f"x_{node_id} = {input_var}")

            self.shape_tracker[node_id] = out_shape

        final_body_node = list(self.graph.predecessors(self.output_node_id))[0]
        self.forward_lines.append(f"return x_{final_body_node}")

        with open(file_path, 'w', encoding='utf-8') as f:
            for imp in sorted(list(self.imports)):
                f.write(f"{imp}\n")
            f.write("\n")
            
            f.write(f"class {class_name}(nn.Module):\n")
            
            f.write("    def __init__(self):\n")
            f.write("        super().__init__()\n")
            if not self.init_lines:
                f.write("        pass\n")
            for line in self.init_lines:
                f.write(f"        {line}\n")
            f.write("\n")
            
            f.write("    def forward(self, x):\n")
            for line in self.forward_lines:
                f.write(f"        {line}\n")
        
        print(f"Successfully generated code for {class_name} in {file_path}")

# Загрузка Данных

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import  transforms, datasets
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import lpips
from PIL import Image

class ImageRestorationDataset(Dataset):
    def __init__(self, degraded_dir, clean_dir, transform=None):
        self.degraded_dir = degraded_dir
        self.clean_dir = clean_dir
        self.transform = transform
        
        self.image_files = sorted(os.listdir(degraded_dir))

    def __len__(self):
        return 10000

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        degraded_img_path = os.path.join(self.degraded_dir, img_name)
        clean_img_path = os.path.join(self.clean_dir, img_name)

        degraded_image = Image.open(degraded_img_path).convert('RGB')
        clean_image = Image.open(clean_img_path).convert('RGB')
        
        if self.transform:
            degraded_image = self.transform(degraded_image)
            clean_image = self.transform(clean_image)
            
        return degraded_image, clean_image

In [ ]:
DATASET_PATH = '../dataset_patches' 
DEGRADED_DIR = os.path.join(DATASET_PATH, 'degraded')
CLEAN_DIR = os.path.join(DATASET_PATH, 'clean')

transform = transforms.Compose([
    transforms.ToTensor()
])

full_dataset = ImageRestorationDataset(
    degraded_dir=DEGRADED_DIR,
    clean_dir=CLEAN_DIR,
    transform=transform
)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
data_train, data_val = random_split(full_dataset, [train_size, val_size])

print(f"Размер обучающей выборки: {len(data_train)}")
print(f"Размер валидационной выборки: {len(data_val)}")

# Служебные функции

In [ ]:
def calculate_fitness(val_lpips, latency, target_latency, w):
    if latency <= 0 or latency == float('inf'):
        return 0.0

    quality_metric = 1.0 - val_lpips
    
    if quality_metric < 0:
        return 0.0

    fitness = quality_metric * (latency / target_latency) ** (-w)
    return fitness

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.5, device="cpu"):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        
        self.l1_loss = nn.L1Loss()
        
        if self.beta > 0:
            self.lpips_loss = lpips.LPIPS(net='alex').to(device)
        else:
            self.lpips_loss = None

    def forward(self, pred, target):
        loss_l1 = self.l1_loss(pred, target)
        total_loss = self.alpha * loss_l1
        
        loss_components = {'pixel': loss_l1.item()}

        if self.lpips_loss is not None:
            pred_lpips = pred.clamp(0.0, 1.0) * 2.0 - 1.0
            target_lpips = target.clamp(0.0, 1.0) * 2.0 - 1.0
            
            loss_lpips_val = torch.mean(self.lpips_loss(pred_lpips, target_lpips))
            total_loss = total_loss + self.beta * loss_lpips_val
            loss_components['perception'] = loss_lpips_val.item()

        loss_components['total'] = total_loss
        
        return loss_components

# Эволюционный алгоритм

In [ ]:
import time
import torch.optim as optim

POPULATION_SIZE = 200
NUM_CYCLES = 100000
TOURNAMENT_SIZE = 20
PRINT_FREQUENCY = 50

INPUT_SHAPE = (3, 256, 256)

available_mutations = ['add', 'remove', 'params', 'add_block', 'step_params']
mutation_weights = [0.2, 0.2, 0.15, 0.15, 0.3]

NUM_EPOCHS=1
BATCH_SIZE=4

GLOBAL_SEED = 42
TRAINING_SEED = 123

LATENCY_HARD_LIMIT = 0.2
TARGET_LATENCY = 0.05
W_FITNESS = 0.02

if __name__ == '__main__':
    set_seed(GLOBAL_SEED)

    base_results_dir = "architectures_results"
    architectures_dir = os.path.join(base_results_dir, "architectures")
    results_file_path = os.path.join(base_results_dir, "results.txt")

    os.makedirs(architectures_dir, exist_ok=True)

    arch_counter = 0

    with open(results_file_path, "w") as f:
        f.write("arch_id,fitness,val_lpips,latency\n")
    
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    print(f"Используемое устройство: {device}")

    CRITERION=CombinedLoss(alpha=1.0, beta=0.2, device=device)

    print("--- Создание начальной популяции ---")
    
    initial_architectures = []
    
    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 3, 'stride': 1})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)

    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 5, 'stride': 1})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)

    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 7, 'stride': 1})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)

    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 3, 'stride': 1, 'dilation': 2})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)

    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 5, 'stride': 1, 'dilation': 2})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)

    base_arch = ArchitectureGraph()
    conv_node = base_arch.add_node(OpType.CONV, params = {'out_channels': 3, 'kernel_size': 7, 'stride': 1, 'dilation': 2})
    base_arch.add_edge(base_arch.input_node, conv_node)
    base_arch.add_edge(conv_node, base_arch.output_node)
    initial_architectures.append(base_arch)
    
    evaluated_hashes = {arch.get_canonical_hash() for arch in initial_architectures}
    population = list(initial_architectures)
    
    filler_needed = POPULATION_SIZE - len(population)
    if filler_needed > 0:
        print(f"Необходимо сгенерировать {filler_needed} дополнительных архитектур...")
        
        while len(population) < POPULATION_SIZE:
            parent_arch = random.choice(initial_architectures)

            mutation_successful = False
            for _ in range(100):
                offspring_candidate = copy.deepcopy(parent_arch)
                mutation_type = random.choices(available_mutations, weights=mutation_weights, k=1)[0]
                
                try:
                    if mutation_type == 'add':
                        offspring_candidate.mutate_add_node()
                    elif mutation_type == 'add_block':
                        offspring_candidate.mutate_add_block()
                    elif mutation_type == 'remove':
                        offspring_candidate.mutate_remove_node()
                    elif mutation_type == 'params':
                        offspring_candidate.mutate_change_params()
                    elif mutation_type == 'step_params':
                        offspring_candidate.mutate_step_params()
                except Exception:
                    continue

                candidate_hash = offspring_candidate.get_canonical_hash()
                
                if (offspring_candidate.is_valid(INPUT_SHAPE) and 
                    offspring_candidate.is_sane() and 
                    candidate_hash not in evaluated_hashes):
                    
                    population.append(offspring_candidate)
                    evaluated_hashes.add(candidate_hash)
                    print(f"  Создана уникальная начальная особь {len(population)}/{POPULATION_SIZE} (мутация: {mutation_type})")
                    mutation_successful = True
                    break
            
            if not mutation_successful:
                print("    WARNING: Не удалось создать валидного потомка за 100 попыток от данного родителя. Повторяем выбор родителя.")

    print(f"\nНачальная популяция из {len(population)} уникальных особей создана.")
    print("-" * 50)

    print("--- Полная оценка начальной популяции ---")
    population_with_results = []
    for i, arch in enumerate(population):
        print(f"\n--- Оценка начальной особи {i+1}/{len(population)} ---")
        try:
            model = build_model_from_graph(arch, INPUT_SHAPE)
            history = train_model(
                num_epochs=NUM_EPOCHS, 
                batch_size=BATCH_SIZE, 
                model=model, 
                optimizer=optim.AdamW(model.parameters()), 
                criterion=CRITERION, 
                train_data=data_train,
                val_data=data_val,
                scheduler=None,
                scheduler_step_on_iter=False,
                predicts_clean=False,
                measure_latency=True,
                time_limit=LATENCY_HARD_LIMIT,
                seed=TRAINING_SEED, 
                device=device
            )

            final_metrics = history[-1]
            val_lpips, latency = final_metrics[2], final_metrics[3]

            fitness = 0.0 if val_lpips == 100.0 else calculate_fitness(val_lpips, latency, TARGET_LATENCY, W_FITNESS)
        
            print(f"Результат: Val LPIPS = {val_lpips:.4f}, Latency = {latency:.4f}s, Fitness = {fitness:.4f}")

            arch_filename = os.path.join(architectures_dir, f"{arch_counter}.txt")
            with open(arch_filename, "w") as f:
                f.write(arch.to_text())
            
            with open(results_file_path, "a") as f:
                f.write(f"{arch_counter},{fitness:.6f},{val_lpips:.6f},{latency:.6f}\n")
            
            arch_counter += 1
            
            population_with_results.append((fitness, val_lpips, latency, arch))
        except Exception as e:
            print(f"!!! Ошибка при оценке особи {i+1}: {e}")

            with open(results_file_path, "a") as f:
                f.write(f"{arch_counter},ERROR,-1.0,999.0\n")
            arch_counter += 1
            
            population_with_results.append((-1.0, 1.0, float('inf'), arch))
    
    print(f"\n{'='*20} НАЧАЛЬНОЕ СОСТОЯНИЕ ПОПУЛЯЦИИ {'='*20}")
    sorted_population = sorted(population_with_results, key=lambda x: x[0], reverse=True)
    for j, (fitness, val_lpips, latency, _) in enumerate(sorted_population):
        print(f"Место {j+1}: Fitness={fitness:.4f} (LPIPS={val_lpips:.4f}, Latency={latency:.4f}s)")

    for cycle in range(NUM_CYCLES):
        print(f"\n{'='*20} ЦИКЛ {cycle + 1}/{NUM_CYCLES} {'='*20}")
        
        offspring_candidate = None
        
        while True:
            tournament_contestants = random.sample(population_with_results, TOURNAMENT_SIZE)
            tournament_winner = max(tournament_contestants, key=lambda x: x[0])
            parent_arch = tournament_winner[3]

            mutation_successful = False
            for _ in range(100):
                candidate = copy.deepcopy(parent_arch)
                mutation_type = random.choices(available_mutations, weights=mutation_weights, k=1)[0]
                
                try:
                    if mutation_type == 'add':
                        candidate.mutate_add_node()
                    elif mutation_type == 'add_block':
                        candidate.mutate_add_block()
                    elif mutation_type == 'remove':
                        candidate.mutate_remove_node()
                    elif mutation_type == 'params':
                        candidate.mutate_change_params()
                except Exception:
                    continue

                candidate_hash = candidate.get_canonical_hash()
                if (candidate.is_valid(INPUT_SHAPE) and 
                    candidate.is_sane() and 
                    candidate_hash not in evaluated_hashes):
                    
                    offspring_candidate = candidate
                    evaluated_hashes.add(candidate_hash)
                    print(f"  Успешно создан уникальный потомок (мутация: {mutation_type})")
                    mutation_successful = True
                    break
            
            if mutation_successful:
                break
            else:
                print("  ПРЕДУПРЕЖДЕНИЕ: Не удалось создать потомка за 100 попыток. Повторяем турнир.")

        print(f"--- Оценка нового потомка ---")
        try:
            model = build_model_from_graph(offspring_candidate, INPUT_SHAPE)
            history = train_model(
                num_epochs=NUM_EPOCHS, 
                batch_size=BATCH_SIZE, 
                model=model, 
                optimizer=optim.AdamW(model.parameters()), 
                criterion=CRITERION, 
                train_data=data_train,
                val_data=data_val,
                scheduler=None,
                scheduler_step_on_iter=False,
                predicts_clean=False,
                measure_latency=True,
                time_limit=LATENCY_HARD_LIMIT,
                seed=TRAINING_SEED, 
                device=device
            )
            final_metrics = history[-1]
            val_lpips, latency = final_metrics[2], final_metrics[3]
            fitness = 0.0 if val_lpips == 100.0 else calculate_fitness(val_lpips, latency, TARGET_LATENCY, W_FITNESS)
            new_individual_with_results = (fitness, val_lpips, latency, offspring_candidate)
            print(f"Результат: Val LPIPS = {val_lpips:.4f}, Latency = {latency:.4f}s, Fitness = {fitness:.4f}")

            arch_filename = os.path.join(architectures_dir, f"{arch_counter}.txt")
            with open(arch_filename, "w") as f:
                f.write(offspring_candidate.to_text())

            with open(results_file_path, "a") as f:
                f.write(f"{arch_counter},{fitness:.6f},{val_lpips:.6f},{latency:.6f}\n")
            
            arch_counter += 1
            
        except Exception as e:
            print(f"!!! Ошибка при оценке нового потомка: {e}")

            with open(results_file_path, "a") as f:
                f.write(f"{arch_counter},ERROR,-1.0,999.0\n")
            arch_counter += 1
            
            new_individual_with_results = (-1.0, 1.0, float('inf'), offspring_candidate)
        
        removed_individual = population_with_results.pop(0)
        population_with_results.append(new_individual_with_results)
        
        print(f"\n  ЗАМЕНА В ПОПУЛЯЦИИ:")
        print(f"  Удалена старейшая особь: Fitness={removed_individual[0]:.4f} (LPIPS={removed_individual[1]:.4f}, Latency={removed_individual[2]:.4f}s)")
        print(f"  Добавлена новая особь:   Fitness={new_individual_with_results[0]:.4f} (LPIPS={new_individual_with_results[1]:.4f}, Latency={new_individual_with_results[2]:.4f}s)")

        if (cycle + 1) % PRINT_FREQUENCY == 0 or (cycle + 1) == NUM_CYCLES:
            print(f"\n--- Текущее состояние популяции (Цикл {cycle + 1}) ---")
            sorted_population = sorted(population_with_results, key=lambda x: x[0], reverse=True)
            for j, (fitness, val_lpips, latency, _) in enumerate(sorted_population):
                print(f"Место {j+1}: Fitness={fitness:.4f} (LPIPS={val_lpips:.4f}, Latency={latency:.4f}s)")


    print(f"\n{'='*20} ГЕНЕТИЧЕСКИЙ АЛГОРИТМ ЗАВЕРШЕН {'='*20}")
    
    final_population = sorted(population_with_results, key=lambda x: x[0], reverse=True)
    final_best_fitness, final_best_lpips, final_best_latency, final_best_arch = final_population[0]
    
    print("Лучшая найденная архитектура:")
    final_best_arch.display_info()
    
    print(f"\nЕе характеристики: Fitness={final_best_fitness:.4f}, Val LPIPS={final_best_lpips:.4f}, Latency={final_best_latency:.4f}s")
    final_best_arch.visualize("Лучшая найденная архитектура")